# 06 — Model B: Cross-Period Next-Day AQI Forecasting

**Research question:** Can an AQI-only forecasting model trained on Indian city data from **2015–2020** forecast **tomorrow’s AQI** on recent observations from **2023–2025**?

This experiment is distinct from Aditri's same-day *pollutant-based* Model A and from the in-period recent-data models. We compare **persistence**, **Ridge**, **Random Forest** and **HistGradientBoosting**. All models use a **shared, pollutant-free feature schema**; no test-period targets are used for fitting. Later, a recent-data-trained reference model is evaluated on the **exact same held-out rows**.

**Important:** The supplied notebooks document paths but do not include the CSV data; this notebook is ready to run once those files are present in your repo. Run cells in order.

## 1. Imports, settings and reproducibility

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'Cleaned'
OUTPUT = ROOT / 'outputs' / 'model_b'
OUTPUT.mkdir(parents=True, exist_ok=True)

# File paths verified from the uploaded notebooks. Edit these if your repo differs.
OLD_CSV = DATA / 'city_day_clean.csv'
NEW_CSV = DATA / 'aqi_2023_2025_clean.csv'
# A valid alternative to NEW_CSV is a daily City/Date/AQI CSV.
RECENT_TRAIN_START = '2023-01-01'
RECENT_TEST_START = '2024-11-01'  # follows the in-period evaluation window
RECENT_TEST_END = '2025-04-30'
HISTORICAL_END = '2020-12-31'

print('Project root:', ROOT)
for p in [OLD_CSV, NEW_CSV]:
    print(('FOUND' if p.exists() else 'MISSING'), p)
if not (OLD_CSV.exists() and NEW_CSV.exists()):
    raise FileNotFoundError('Place both cleaned CSVs at the paths shown above, then rerun. Do not substitute pre-generated feature tables.')

Project root: d:\IISER\7th sem\idc409(data science)\AQI_project_idc409\AQI-prediction-India
FOUND d:\IISER\7th sem\idc409(data science)\AQI_project_idc409\AQI-prediction-India\data\Cleaned\city_day_clean.csv
FOUND d:\IISER\7th sem\idc409(data science)\AQI_project_idc409\AQI-prediction-India\data\Cleaned\aqi_2023_2025_clean.csv


## 2. Load datasets

The older data comes from `data/Cleaned/city_day_clean.csv` (Aditri's `01_eda.ipynb`); the newer data from `data/Cleaned/aqi_2023_2025_clean.csv` (your `03_feature_engineering_2023_25.ipynb`). No pollutant variables are necessary for this **AQI-only** experiment.

In [3]:
old_raw = pd.read_csv(OLD_CSV, low_memory=False)
new_raw = pd.read_csv(NEW_CSV, low_memory=False)
print('Historical raw shape:', old_raw.shape)
print('Recent raw shape:', new_raw.shape)
print('Old columns:', list(old_raw.columns))
print('New columns:', list(new_raw.columns))

Historical raw shape: (24801, 23)
Recent raw shape: (235005, 7)
Old columns: ['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'AQI', 'AQI_Bucket', 'Year', 'Month', 'Day', 'Weekday', 'Is_Weekend', 'Season', 'Is_Winter', 'Is_Diwali_Period']
New columns: ['Date', 'state', 'City', 'number_of_monitoring_stations', 'prominent_pollutants', 'AQI', 'AQI_Bucket']


## 3. Standardize City, Date and AQI; verify coverage and overlap

## 4. Shared, leakage-safe daily feature engineering

At each date **t**, `AQI` is the observed current-day AQI, and the target is **AQI(t+1)**. Previous-day lags and rolling statistics are built from a city-specific daily grid, so absent dates remain `NaN` rather than becoming the previous row. No future target is used as a feature. Date features are defined identically across datasets. Festival and pollutant features are **intentionally excluded** because comparable measurements are not guaranteed across both sources.

**Operational assumption:** today's finalized AQI is available when forecasting tomorrow. If you need an early-morning forecast, change that assumption and remove same-day AQI.

## 5. Define a strict cross-period test and a fair recent-data comparator

**Model B train**: historical samples with target dates no later than 2020-12-31. **Test**: newer samples whose *target date* is 2024-11-01 through 2025-04-30. **Recent-trained reference**: newer samples whose target dates are before 2024-11-01. This is a different training distribution but is scored on the **same test rows**.

Only cities appearing in the historical, recent-training and test partitions are retained. Reuse the exact target-date window across all methods.

## 6. Train persistence, Ridge, Random Forest and HistGradientBoosting

## 7. Regression evaluation and exported predictions

## 8. Visualize actual vs predicted AQI and city-wise errors

## 9. Fair comparison: recent-trained models on the identical held-out rows

To assess **distribution shift**, fit the *same* feature definitions and model configurations on the **recent training window only**, then score on the already-defined test set. Do not copy scores from `04_modelling_2023_25.ipynb`: that notebook may use extra pollutant/station features and a different subset of cities. We compare training periods only after aligning the test observations and input features.